## Gold Aggregates
Business-ready KPI tables for the dashboard, built from the star schema
(`fact_video_daily_stats` + dimensions).


In [0]:
%run ./00_config

### agg_category_daily

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {gold['agg_category_daily']}
        (date_key, full_date, category_id, category_name,
         videos_in_trending, views_gained, like_rate, _updated_at)
    SELECT
        f.date_key,
        d.full_date,
        f.category_id,
        c.category_name,
        COUNT(DISTINCT f.video_id),
        SUM(f.views_delta),
        ROUND(SUM(f.likes) / NULLIF(SUM(f.views), 0), 4),
        current_timestamp()
    FROM {gold['fact_video_daily_stats']} AS f
    JOIN {gold['dim_date']}     AS d ON f.date_key    = d.date_key
    JOIN {gold['dim_category']} AS c ON f.category_id = c.category_id
    GROUP BY f.date_key, d.full_date, f.category_id, c.category_name
""")

### agg_channel_summary

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {gold['agg_channel_summary']}
        (channel_key, channel_title, videos_trended, avg_days_per_video, total_views, _updated_at)
    WITH per_video AS (
        SELECT channel_key, video_id, COUNT(*) AS days_in_trending, MAX(views) AS peak_views
        FROM {gold['fact_video_daily_stats']}
        GROUP BY channel_key, video_id
    )
    SELECT p.channel_key, c.channel_title, COUNT(*),
           ROUND(AVG(p.days_in_trending), 2), SUM(p.peak_views), current_timestamp()
    FROM per_video AS p
    JOIN {gold['dim_channel']} AS c ON p.channel_key = c.channel_key
    GROUP BY p.channel_key, c.channel_title
""")